# Training Model NER — UTS NLP

Tujuan:
1. Memeriksa dan memvalidasi anotasi entitas.
2. Membuat baseline NER.
3. Melatih atau mengadaptasi model NER.
4. Mengevaluasi precision, recall, dan F1-score.
5. Menyimpan model untuk digunakan pada API.

In [1]:
!pip install -q spacy pandas scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 88.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from google.colab import files
import pandas as pd
import json
import os
import random
import shutil
from collections import Counter, defaultdict

uploaded = files.upload()

CSV_PATH = next(
    name for name in uploaded
    if name.lower().endswith(".csv")
)

df = pd.read_csv(
    CSV_PATH,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

print("File:", CSV_PATH)
print("Jumlah baris:", len(df))
print("Kolom:", df.columns.tolist())

display(df.head(3))

Saving NER_annotation_queue_draft_filled (1).csv to NER_annotation_queue_draft_filled (1).csv
File: NER_annotation_queue_draft_filled (1).csv
Jumlah baris: 120
Kolom: ['original_id', 'title', 'body', 'clipping', 'source_name', 'date_published', 'split', 'entities_json']


,original_id,title,body,clipping,source_name,date_published,split,entities_json
0,b0581ce5a41b994f4f91d383209382023ef6d8d6835ffa...,FBM KLCI charts slight gain at midday | KLSE S...,KUALA LUMPUR: Malaysia's benchmark index lagge...,Hong Leong Bank,klsescreener.com,05/10/2026 13.00.00,train,"[{""text"": ""FBM KLCI"", ""label"": ""ORG"", ""field"":..."
1,6b505436351b490f0b76aa84ceec89ee515ea2e9194d54...,Diplomasi Sukan Anwar Letakkan Malaysia Kembal...,Pendekatan Perdana Menteri Datuk Seri Anwar Ib...,Hong Leong Bank,malaysiadateline.com,04/10/2026 14.00.00,train,"[{""text"": ""Malaysia"", ""label"": ""GPE"", ""field"":..."
2,1da8622c4a60966cc2f5238621cc0e65ca9750fec72726...,World Bank Supports Malaysia’s Possible Integr...,KUALA LUMPUR (Oct 6): The World Bank has agree...,CIMB Group Malaysia,klse.i3investor.com,07/10/2026 10.27.43,train,"[{""text"": ""World Bank"", ""label"": ""ORG"", ""field..."


In [3]:
required_columns = {
    "original_id",
    "title",
    "body",
    "clipping",
    "split",
    "entities_json"
}

assert required_columns.issubset(df.columns), (
    f"Kolom kurang: {required_columns - set(df.columns)}"
)

print("Distribusi split:")
print(df["split"].value_counts())

print("\nJumlah entities_json kosong:")
print(df["entities_json"].str.strip().eq("").sum())

print("\nContoh anotasi:")
print(df.loc[
    df["entities_json"].str.strip().ne(""),
    ["title", "split", "entities_json"]
].head(2).to_string(index=False))

Distribusi split:
split
train         84
validation    18
test          18
Name: count, dtype: int64

Jumlah entities_json kosong:
1

Contoh anotasi:
                                                                           title split                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [4]:
import spacy

tokenizer = spacy.load("en_core_web_sm")

records = []
invalid_rows = []
blank_rows = []

for _, row in df.iterrows():
    raw_json = row["entities_json"].strip()

    # Anotasi kosong tidak dianggap sebagai artikel tanpa entitas.
    if not raw_json:
        blank_rows.append(row["original_id"])
        continue

    try:
        entities = json.loads(raw_json)
        if not isinstance(entities, list):
            raise ValueError("entities_json harus berupa list")

        title = row["title"]
        body = row["body"]
        full_text = title + "\n" + body

        doc = tokenizer.make_doc(full_text)
        spans = []
        problems = []

        for ent in entities:
            field = ent.get("field")
            source_text = {
                "title": title,
                "body": body
            }.get(field)

            if source_text is None:
                problems.append(f"Field tidak valid: {field}")
                continue

            start = int(ent["start_char"])
            end = int(ent["end_char"])
            label = ent["label"]
            entity_text = ent["text"]

            # Validasi teks sumber dan offset asli.
            if not (0 <= start < end <= len(source_text)):
                problems.append(f"Offset di luar teks: {entity_text}")
                continue

            if source_text[start:end] != entity_text:
                problems.append(f"Teks tidak cocok: {entity_text}")
                continue

            # Geser offset body ke posisi teks gabungan.
            offset = 0 if field == "title" else len(title) + 1
            combined_start = start + offset
            combined_end = end + offset

            # Memastikan rentang sesuai dengan batas token spaCy.
            span = doc.char_span(
                combined_start,
                combined_end,
                label=label,
                alignment_mode="strict"
            )

            if span is None:
                problems.append(
                    f"Rentang tidak sejajar dengan token: {entity_text}"
                )
                continue

            spans.append((combined_start, combined_end, label))

        # Memastikan tidak ada rentang yang saling tumpang tindih.
        sorted_spans = sorted(spans)
        for a, b in zip(sorted_spans, sorted_spans[1:]):
            if b[0] < a[1]:
                problems.append("Ditemukan entitas yang tumpang tindih")

        if problems:
            invalid_rows.append({
                "original_id": row["original_id"],
                "split": row["split"],
                "problems": problems
            })
            continue

        records.append({
            "original_id": row["original_id"],
            "split": row["split"].strip().lower(),
            "text": full_text,
            "spans": sorted_spans
        })

    except Exception as e:
        invalid_rows.append({
            "original_id": row["original_id"],
            "split": row["split"],
            "problems": [str(e)]
        })

print("Artikel siap diproses:", len(records))
print("Artikel tanpa anotasi:", len(blank_rows))
print("Artikel dengan masalah:", len(invalid_rows))

if invalid_rows:
    display(pd.DataFrame(invalid_rows).head(20))

Artikel siap diproses: 89
Artikel tanpa anotasi: 1
Artikel dengan masalah: 30


,original_id,split,problems
0,1da8622c4a60966cc2f5238621cc0e65ca9750fec72726...,train,[Rentang tidak sejajar dengan token: US]
1,03e3b4d767950fac90f23f15fae288b322157c95a0dcba...,train,"[Rentang tidak sejajar dengan token: MAYBANK, ..."
2,14ac89c0642bc07a6ea2829e5c7d6a41bde6f884413305...,train,[Rentang tidak sejajar dengan token: US]
3,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,train,"[Rentang tidak sejajar dengan token: US, Renta..."
4,97bd3c7631ceef97b005f4f3dc4c5a6a5063ee644df2fc...,train,[Rentang tidak sejajar dengan token: KUALA LUM...
5,57749ef7f659d012743a65a3e09d1d9f1e6c595f984a2f...,train,"[Rentang tidak sejajar dengan token: MAYBANK, ..."
6,c353c01e7d832da84b098f380e44d19b1b59677890c7a1...,train,"[Rentang tidak sejajar dengan token: 2027, Ren..."
7,30c1808b3b09dd97cd94161c4e25e998614984b3cfd964...,train,[Rentang tidak sejajar dengan token: US]
8,13683027559165475c02bbc6ed21b19f78fedeee3416a4...,train,"[Rentang tidak sejajar dengan token: US, Renta..."
9,b6b9b504c6199322e0b6997d968b2c6c371d1e69a790a6...,train,[Rentang tidak sejajar dengan token: US]


In [5]:
import pandas as pd
from collections import Counter

# Ringkasan jenis masalah
issue_counts = Counter(
    problem.split(":")[0]
    for row in invalid_rows
    for problem in row["problems"]
)

print("Jumlah artikel bermasalah:", len(invalid_rows))
print("\nJenis masalah:")

for problem, count in issue_counts.items():
    print(f"- {problem}: {count}")

# Buat tabel error
error_df = pd.DataFrame(invalid_rows)

# Ambil judul artikel dari dataframe asli
title_lookup = (
    df.assign(original_id=df["original_id"].astype(str))
      .drop_duplicates("original_id")
      .set_index("original_id")["title"]
)

error_df["original_id"] = error_df["original_id"].astype(str)
error_df["title"] = error_df["original_id"].map(title_lookup)

# Tampilkan hasil
display(
    error_df[
        ["original_id", "split", "title", "problems"]
    ].head(15)
)

Jumlah artikel bermasalah: 30

Jenis masalah:
- Rentang tidak sejajar dengan token: 80


,original_id,split,title,problems
0,1da8622c4a60966cc2f5238621cc0e65ca9750fec72726...,train,World Bank Supports Malaysia’s Possible Integr...,[Rentang tidak sejajar dengan token: US]
1,03e3b4d767950fac90f23f15fae288b322157c95a0dcba...,train,马股周二情绪稍微乐观，跟随区域市场积极走势，综指小幅收涨,"[Rentang tidak sejajar dengan token: MAYBANK, ..."
2,14ac89c0642bc07a6ea2829e5c7d6a41bde6f884413305...,train,Mesiniaga Wins Two-year RM32.3m Contract Exten...,[Rentang tidak sejajar dengan token: US]
3,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,train,Optimistic outlook for equities in fourth quarter,"[Rentang tidak sejajar dengan token: US, Renta..."
4,97bd3c7631ceef97b005f4f3dc4c5a6a5063ee644df2fc...,train,High Court to rule Oct 23 on Maritime Network’...,[Rentang tidak sejajar dengan token: KUALA LUM...
5,57749ef7f659d012743a65a3e09d1d9f1e6c595f984a2f...,train,Maybank emerges as top winner at inaugural Bur...,"[Rentang tidak sejajar dengan token: MAYBANK, ..."
6,c353c01e7d832da84b098f380e44d19b1b59677890c7a1...,train,Markets brace for tighter Fed path: CIMB Secur...,"[Rentang tidak sejajar dengan token: 2027, Ren..."
7,30c1808b3b09dd97cd94161c4e25e998614984b3cfd964...,train,Felda’s Road Ahead in the Spotlight,[Rentang tidak sejajar dengan token: US]
8,13683027559165475c02bbc6ed21b19f78fedeee3416a4...,train,India’s Palm Oil Tax Cut Could Boost Earnings ...,"[Rentang tidak sejajar dengan token: US, Renta..."
9,b6b9b504c6199322e0b6997d968b2c6c371d1e69a790a6...,train,World Bank Urges Malaysia to Adjust SME Tax St...,[Rentang tidak sejajar dengan token: US]


In [6]:

# tokenizer = spacy.load("en_core_web_sm")

debug_rows = []

for _, row in df.iterrows():
    raw_json = row["entities_json"].strip()

    if not raw_json:
        continue

    title = row["title"]
    body = row["body"]

    doc = tokenizer.make_doc(title + "\n" + body)

    for ent in json.loads(raw_json):
        field = ent.get("field")
        source_text = {
            "title": title,
            "body": body
        }.get(field)

        if source_text is None:
            continue

        start = int(ent["start_char"])
        end = int(ent["end_char"])
        text = ent["text"]

        if not (0 <= start < end <= len(source_text)):
            continue

        if source_text[start:end] != text:
            continue

        offset = 0 if field == "title" else len(title) + 1

        span = doc.char_span(
            start + offset,
            end + offset,
            label=ent["label"],
            alignment_mode="strict"
        )

        if span is None:
            debug_rows.append({
                "original_id": row["original_id"],
                "title": title,
                "field": field,
                "entity_text": text,
                "label": ent["label"],
                "context": source_text[
                    max(0, start - 30):
                    min(len(source_text), end + 30)
                ]
            })

debug_df = pd.DataFrame(debug_rows)

print("Jumlah rentang yang gagal:", len(debug_df))
display(debug_df.head(20))

Jumlah rentang yang gagal: 80


,original_id,title,field,entity_text,label,context
0,1da8622c4a60966cc2f5238621cc0e65ca9750fec72726...,World Bank Supports Malaysia’s Possible Integr...,body,US,GPE,roup is exploring a potential US$200 million s...
1,03e3b4d767950fac90f23f15fae288b322157c95a0dcba...,马股周二情绪稍微乐观，跟随区域市场积极走势，综指小幅收涨,body,MAYBANK,ORG,板能源股）跌8仙至19.12令吉。 银行股普遍走低，马银行（MAYBANK，1155）跌3仙...
2,03e3b4d767950fac90f23f15fae288b322157c95a0dcba...,马股周二情绪稍微乐观，跟随区域市场积极走势，综指小幅收涨,body,CIMB,ORG,银行（PBBANK，1295）跌4仙4.67令吉，联昌国际（CIMB，1023）跌3仙至7....
3,14ac89c0642bc07a6ea2829e5c7d6a41bde6f884413305...,Mesiniaga Wins Two-year RM32.3m Contract Exten...,body,US,GPE,roup is exploring a potential US$200 million s...
4,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,Optimistic outlook for equities in fourth quarter,body,US,GPE,"r, sustained oil prices above US$100 a barrel,..."
5,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,Optimistic outlook for equities in fourth quarter,body,US,GPE,s expected to decline towards US$75 to US$85 p...
6,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,Optimistic outlook for equities in fourth quarter,body,US,GPE,d to decline towards US$75 to US$85 per barrel...
7,97bd3c7631ceef97b005f4f3dc4c5a6a5063ee644df2fc...,High Court to rule Oct 23 on Maritime Network’...,body,KUALA LUMPUR,GPE,"=""wp-block-paragraph""><strong>KUALA LUMPUR</st..."
8,97bd3c7631ceef97b005f4f3dc4c5a6a5063ee644df2fc...,High Court to rule Oct 23 on Maritime Network’...,body,2005,DATE,urrent account with RHB since 2005.</p> <p cl...
9,97bd3c7631ceef97b005f4f3dc4c5a6a5063ee644df2fc...,High Court to rule Oct 23 on Maritime Network’...,body,"July 2, 2025",DATE,"ismissed by the High Court on July 2, 2025.</p..."


In [7]:
import json
import pandas as pd

diagnostics = []

for _, row in df.iterrows():
    raw_json = str(row["entities_json"]).strip()

    if not raw_json:
        continue

    title = row["title"]
    body = row["body"]
    full_text = title + "\n" + body

    doc = tokenizer.make_doc(full_text)

    for ent in json.loads(raw_json):
        field = ent.get("field")

        source_text = {
            "title": title,
            "body": body
        }.get(field)

        if source_text is None:
            continue

        start = int(ent["start_char"])
        end = int(ent["end_char"])
        entity_text = ent["text"]

        # Pastikan offset cocok dengan teks sumber terlebih dahulu.
        if not (0 <= start < end <= len(source_text)):
            continue

        if source_text[start:end] != entity_text:
            continue

        offset = 0 if field == "title" else len(title) + 1
        combined_start = start + offset
        combined_end = end + offset

        span = doc.char_span(
            combined_start,
            combined_end,
            label=ent["label"],
            alignment_mode="strict"
        )

        if span is not None:
            continue

        # Cari token yang berada di sekitar rentang bermasalah.
        nearby_tokens = [
            f"{t.text!r} [{t.idx}:{t.idx + len(t)}]"
            for t in doc
            if t.idx < combined_end + 10
            and t.idx + len(t) > combined_start - 10
        ]

        expanded = doc.char_span(
            combined_start,
            combined_end,
            label=ent["label"],
            alignment_mode="expand"
        )

        diagnostics.append({
            "original_id": row["original_id"],
            "split": row["split"],
            "field": field,
            "entity_text": entity_text,
            "label": ent["label"],
            "start": start,
            "end": end,
            "source_slice": source_text[start:end],
            "expanded_candidate": (
                expanded.text if expanded else None
            ),
            "nearby_tokens": " | ".join(nearby_tokens)
        })

diagnostic_df = pd.DataFrame(diagnostics)

print("Jumlah rentang bermasalah:", len(diagnostic_df))

display(
    diagnostic_df[
        [
            "entity_text",
            "label",
            "source_slice",
            "expanded_candidate",
            "nearby_tokens"
        ]
    ].head(20)
)

Jumlah rentang bermasalah: 80


,entity_text,label,source_slice,expanded_candidate,nearby_tokens
0,US,GPE,US,US$,'potential' [3662:3671] | 'US$' [3672:3675] | ...
1,MAYBANK,ORG,MAYBANK,银行股普遍走低，马银行（MAYBANK，1155）跌3仙至9.97令吉，大众银行（PBBAN...,'银行股普遍走低，马银行（MAYBANK，1155）跌3仙至9.97令吉，大众银行（PBBA...
2,CIMB,ORG,CIMB,银行股普遍走低，马银行（MAYBANK，1155）跌3仙至9.97令吉，大众银行（PBBAN...,'银行股普遍走低，马银行（MAYBANK，1155）跌3仙至9.97令吉，大众银行（PBBA...
3,US,GPE,US,US$,'potential' [2780:2789] | 'US$' [2790:2793] | ...
4,US,GPE,US,US$,'prices' [989:995] | 'above' [996:1001] | 'US$...
5,US,GPE,US,US$,'decline' [4659:4666] | 'towards' [4667:4674] ...
6,US,GPE,US,US$,'US$' [4675:4678] | '75' [4678:4680] | 'to' [4...
7,KUALA LUMPUR,GPE,KUALA LUMPUR,KUALA LUMPUR</strong,"'paragraph""><strong' [97:115] | '>' [115:116] ..."
8,2005,DATE,2005,2005.</p,'RHB' [1511:1514] | 'since' [1515:1520] | '200...
9,"July 2, 2025",DATE,"July 2, 2025","July 2, 2025.</p",'Court' [2120:2125] | 'on' [2126:2128] | 'July...


In [8]:
import re
import html

def clean_html_with_mapping(raw_text):
    """
    Menghapus tag HTML, mendekode HTML entity,
    dan menyimpan pemetaan offset teks mentah ke teks bersih.
    """
    raw_text = str(raw_text)
    output = []
    boundary = [0] * (len(raw_text) + 1)
    i = 0

    while i < len(raw_text):
        boundary[i] = len(output)

        # Hapus komentar HTML
        if raw_text.startswith("<!--", i):
            end = raw_text.find("-->", i + 4)

            if end != -1:
                end += 3
                if output and not output[-1].isspace():
                    output.append(" ")

                for k in range(i + 1, end + 1):
                    boundary[k] = len(output)

                i = end
                continue

        # Hapus tag HTML seperti <p>, </strong>, dan <a ...>
        if raw_text[i] == "<":
            match = re.match(r"<[^>]*>", raw_text[i:], flags=re.S)

            if match:
                end = i + match.end()

                if output and not output[-1].isspace():
                    output.append(" ")

                for k in range(i + 1, end + 1):
                    boundary[k] = len(output)

                i = end
                continue

        # Dekode entity seperti &nbsp; dan &amp;
        if raw_text[i] == "&":
            match = re.match(
                r"&(?:#[xX][0-9a-fA-F]+|#\d+|[a-zA-Z][a-zA-Z0-9]+);",
                raw_text[i:]
            )

            if match:
                end = i + match.end()
                encoded = match.group()
                decoded = html.unescape(encoded)

                if decoded != encoded:
                    start_output = len(output)

                    for k in range(i, end):
                        boundary[k] = start_output

                    output.extend(decoded.replace("\xa0", " "))
                    boundary[end] = len(output)

                    i = end
                    continue

        # Salin karakter biasa
        boundary[i] = len(output)
        output.append(raw_text[i])
        boundary[i + 1] = len(output)
        i += 1

    return "".join(output), boundary

In [9]:
alignment_after_cleaning = []

for _, row in df.iterrows():
    raw_json = str(row["entities_json"]).strip()

    if not raw_json:
        continue

    title, title_map = clean_html_with_mapping(row["title"])
    body, body_map = clean_html_with_mapping(row["body"])

    clean_text = title + "\n" + body
    doc = tokenizer.make_doc(clean_text)

    for ent in json.loads(raw_json):
        field = ent.get("field")
        source_raw = {
            "title": row["title"],
            "body": row["body"]
        }.get(field)

        mapping = {
            "title": title_map,
            "body": body_map
        }.get(field)

        source_clean = {
            "title": title,
            "body": body
        }.get(field)

        if source_raw is None:
            continue

        start = int(ent["start_char"])
        end = int(ent["end_char"])

        if not (0 <= start < end <= len(source_raw)):
            continue

        if source_raw[start:end] != ent["text"]:
            continue

        new_start = mapping[start]
        new_end = mapping[end]

        # Pastikan pembersihan mempertahankan teks entitas.
        if source_clean[new_start:new_end] != ent["text"]:
            alignment_after_cleaning.append({
                "original_id": row["original_id"],
                "entity": ent["text"],
                "label": ent["label"],
                "issue": "Offset berubah saat HTML dibersihkan"
            })
            continue

        offset = 0 if field == "title" else len(title) + 1

        span = doc.char_span(
            new_start + offset,
            new_end + offset,
            label=ent["label"],
            alignment_mode="strict"
        )

        if span is None:
            alignment_after_cleaning.append({
                "original_id": row["original_id"],
                "entity": ent["text"],
                "label": ent["label"],
                "issue": "Masih tidak sejajar dengan token"
            })

print("Rentang yang masih bermasalah:",
      len(alignment_after_cleaning))

display(pd.DataFrame(alignment_after_cleaning).head(20))

Rentang yang masih bermasalah: 63


,original_id,entity,label,issue
0,1da8622c4a60966cc2f5238621cc0e65ca9750fec72726...,US,GPE,Masih tidak sejajar dengan token
1,03e3b4d767950fac90f23f15fae288b322157c95a0dcba...,MAYBANK,ORG,Masih tidak sejajar dengan token
2,03e3b4d767950fac90f23f15fae288b322157c95a0dcba...,CIMB,ORG,Masih tidak sejajar dengan token
3,14ac89c0642bc07a6ea2829e5c7d6a41bde6f884413305...,US,GPE,Masih tidak sejajar dengan token
4,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,US,GPE,Masih tidak sejajar dengan token
5,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,US,GPE,Masih tidak sejajar dengan token
6,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,US,GPE,Masih tidak sejajar dengan token
7,c353c01e7d832da84b098f380e44d19b1b59677890c7a1...,2027,DATE,Masih tidak sejajar dengan token
8,c353c01e7d832da84b098f380e44d19b1b59677890c7a1...,2027,DATE,Masih tidak sejajar dengan token
9,c353c01e7d832da84b098f380e44d19b1b59677890c7a1...,US,GPE,Masih tidak sejajar dengan token


Mengelompokkan seluruh masalah

In [10]:
remaining_df = pd.DataFrame(alignment_after_cleaning)

print("Jumlah rentang bermasalah:",
      len(remaining_df))

print("\nRingkasan berdasarkan jenis masalah:")
display(
    remaining_df.groupby(
        ["issue", "label"],
        dropna=False
    ).size().reset_index(name="jumlah")
)

print("\nEntitas yang paling sering bermasalah:")
display(
    remaining_df.groupby(
        ["entity", "label", "issue"],
        dropna=False
    ).size()
    .reset_index(name="jumlah")
    .sort_values("jumlah", ascending=False)
    .head(20)
)

Jumlah rentang bermasalah: 63

Ringkasan berdasarkan jenis masalah:


,issue,label,jumlah
0,Masih tidak sejajar dengan token,DATE,11
1,Masih tidak sejajar dengan token,GPE,31
2,Masih tidak sejajar dengan token,ORG,6
3,Offset berubah saat HTML dibersihkan,DATE,11
4,Offset berubah saat HTML dibersihkan,GPE,2
5,Offset berubah saat HTML dibersihkan,ORG,2



Entitas yang paling sering bermasalah:


,entity,label,issue,jumlah
11,US,GPE,Masih tidak sejajar dengan token,31
3,2026,DATE,Offset berubah saat HTML dibersihkan,10
4,2027,DATE,Masih tidak sejajar dengan token,3
8,CIMB,ORG,Masih tidak sejajar dengan token,3
10,MAYBANK,ORG,Masih tidak sejajar dengan token,3
9,Indonesia,GPE,Offset berubah saat HTML dibersihkan,2
1,2020,DATE,Masih tidak sejajar dengan token,2
2,2022,DATE,Masih tidak sejajar dengan token,2
6,29 Sept,DATE,Masih tidak sejajar dengan token,2
12,maybank,ORG,Offset berubah saat HTML dibersihkan,2


**riview anotasi**





In [11]:
import json
import pandas as pd
from google.colab import files

# Ambil daftar kombinasi artikel, entitas, dan label
problem_keys = set(
    (
        str(row["original_id"]),
        str(row["entity"]),
        str(row["label"])
    )
    for _, row in remaining_df.iterrows()
)

review_rows = []

for _, row in df.iterrows():
    raw_json = str(row["entities_json"]).strip()

    if not raw_json:
        continue

    original_id = str(row["original_id"])

    for ent in json.loads(raw_json):
        entity_text = str(ent.get("text", ""))
        label = str(ent.get("label", ""))

        key = (original_id, entity_text, label)

        if key not in problem_keys:
            continue

        field = ent.get("field")
        source_text = {
            "title": str(row["title"]),
            "body": str(row["body"])
        }.get(field)

        if source_text is None:
            continue

        start = int(ent["start_char"])
        end = int(ent["end_char"])

        valid_offset = (
            0 <= start < end <= len(source_text)
        )

        actual_text = (
            source_text[start:end]
            if valid_offset else ""
        )

        context_start = max(0, start - 80)
        context_end = min(len(source_text), end + 80)

        review_rows.append({
            "original_id": original_id,
            "title": row["title"],
            "split": row["split"],
            "field": field,
            "entity_text": entity_text,
            "label": label,
            "start_char": start,
            "end_char": end,
            "source_text_matches": actual_text == entity_text,
            "actual_text": actual_text,
            "context": source_text[context_start:context_end],
            "review_decision": ""
        })

review_df = pd.DataFrame(review_rows)

print("Jumlah anotasi untuk diperiksa:", len(review_df))
display(review_df.head(15))

Jumlah anotasi untuk diperiksa: 131


,original_id,title,split,field,entity_text,label,start_char,end_char,source_text_matches,actual_text,context,review_decision
0,1da8622c4a60966cc2f5238621cc0e65ca9750fec72726...,World Bank Supports Malaysia’s Possible Integr...,train,body,US,GPE,3595,3597,True,US,"ver a disputed quit rent hike. Oct 07, 2026 CI...",
1,03e3b4d767950fac90f23f15fae288b322157c95a0dcba...,马股周二情绪稍微乐观，跟随区域市场积极走势，综指小幅收涨,train,body,MAYBANK,ORG,400,407,True,MAYBANK,ENAGA，5347，主板公用事业股）跌10仙至12.90令吉，国油贸易（PETDAG，56...,
2,03e3b4d767950fac90f23f15fae288b322157c95a0dcba...,马股周二情绪稍微乐观，跟随区域市场积极走势，综指小幅收涨,train,body,CIMB,ORG,456,460,True,CIMB,8仙至19.12令吉。 银行股普遍走低，马银行（MAYBANK，1155）跌3仙至9.97令...,
3,14ac89c0642bc07a6ea2829e5c7d6a41bde6f884413305...,Mesiniaga Wins Two-year RM32.3m Contract Exten...,train,body,US,GPE,2721,2723,True,US,"ver a disputed quit rent hike. Oct 07, 2026 CI...",
4,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,Optimistic outlook for equities in fourth quarter,train,body,US,GPE,952,954,True,US,ng renewed attention to domestic catalysts. Ho...,
5,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,Optimistic outlook for equities in fourth quarter,train,body,US,GPE,3096,3098,True,US,remained clouded by geopolitical and macroecon...,
6,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,Optimistic outlook for equities in fourth quarter,train,body,US,GPE,3544,3546,True,US,noted. The research house said the next major...,
7,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,Optimistic outlook for equities in fourth quarter,train,body,US,GPE,3611,3613,True,US,"atch was the US midterm elections on Nov 3, wh...",
8,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,Optimistic outlook for equities in fourth quarter,train,body,US,GPE,4625,4627,True,US,"ons in the Middle East, with Brent crude oil p...",
9,997bc7a9d666dcfadec7e95631d846aed1678416d7397d...,Optimistic outlook for equities in fourth quarter,train,body,US,GPE,4634,4636,True,US,"e Middle East, with Brent crude oil prices exp...",


In [12]:
review_path = "/content/ner_annotation_review.csv"

review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

files.download(review_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Memperbaiki tokenizer

In [13]:
import spacy
import re
import html
import json
import pandas as pd

from spacy.util import compile_infix_regex
from spacy.symbols import ORTH

nlp = spacy.load("en_core_web_sm")

def configure_tokenizer(nlp):
    infixes = list(nlp.Defaults.infixes)

    # Memisahkan pola seperti end-2027
    hyphen_year_pattern = r"(?<=[A-Za-z])-(?=\d)"

    if hyphen_year_pattern not in infixes:
        infixes.append(hyphen_year_pattern)

    nlp.tokenizer.infix_finditer = (
        compile_infix_regex(infixes).finditer
    )

    # Memisahkan Sept dan titik penutup kalimat
    nlp.tokenizer.add_special_case(
        "Sept.",
        [
            {ORTH: "Sept"},
            {ORTH: "."}
        ]
    )

    return nlp

nlp = configure_tokenizer(nlp)

print([
    token.text
    for token in nlp.make_doc(
        "Rates may change by end-2027, including 29 Sept."
    )
])

['Rates', 'may', 'change', 'by', 'end', '-', '2027', ',', 'including', '29', 'Sept', '.']


Membersihkan HTML sambil memetakan offset

In [14]:
def strip_html_with_mapping(raw):
    raw = str(raw)

    output = []
    boundary = [0] * (len(raw) + 1)
    inside_tag = [False] * len(raw)

    block_tags = {
        "p", "div", "br", "hr", "tr", "td", "th",
        "li", "ul", "ol", "section", "article",
        "h1", "h2", "h3", "h4", "table", "blockquote"
    }

    i = 0

    while i < len(raw):
        boundary[i] = len(output)

        # Buang komentar HTML
        if raw.startswith("<!--", i):
            end = raw.find("-->", i + 4)
            j = len(raw) if end < 0 else end + 3

            for k in range(i, j):
                inside_tag[k] = True

            for k in range(i + 1, j + 1):
                boundary[k] = len(output)

            i = j
            continue

        # Buang tag HTML, tetapi jangan menghapus teks di antara tag
        if raw[i] == "<":
            match = re.match(
                r"<\s*(/?)\s*([a-zA-Z][\w:-]*)\b[^>]*>",
                raw[i:],
                flags=re.S
            )

            if match:
                j = i + match.end()
                tag = match.group(2).lower()
                before = len(output)

                if (
                    tag in block_tags
                    and output
                    and not output[-1].isspace()
                ):
                    output.append(" ")

                for k in range(i, j):
                    inside_tag[k] = True

                for k in range(i + 1, j):
                    boundary[k] = len(output)

                boundary[i] = before
                boundary[j] = len(output)

                i = j
                continue

        # Dekode HTML entity seperti &nbsp; dan &amp;
        if raw[i] == "&":
            match = re.match(
                r"&(?:#[xX][0-9a-fA-F]+|#\d+|"
                r"[a-zA-Z][a-zA-Z0-9]+);",
                raw[i:]
            )

            if match:
                j = i + match.end()

                decoded = html.unescape(
                    match.group()
                ).replace("\xa0", " ")

                if decoded != match.group():
                    start = len(output)
                    output.extend(decoded)

                    for k in range(i + 1, j):
                        boundary[k] = start

                    boundary[i] = start
                    boundary[j] = len(output)

                    i = j
                    continue

        output.append(raw[i])
        boundary[i + 1] = len(output)
        i += 1

    return "".join(output), boundary, inside_tag

Menyiapkan ulang data training

In [15]:
!pip install -q langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [16]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42

def guess_language(text):
    sample = re.sub(r"\s+", " ", str(text)).strip()[:4000]

    if len(sample) < 80:
        return "unknown"

    try:
        return detect(sample)
    except Exception:
        return "unknown"


df_work = df.copy()

cleaned_rows = []

for _, row in df_work.iterrows():
    title_clean, _, _ = strip_html_with_mapping(
        row["title"]
    )

    body_clean, _, _ = strip_html_with_mapping(
        row["body"]
    )

    text = title_clean + "\n" + body_clean

    cleaned_rows.append({
        **row.to_dict(),
        "title_clean": title_clean,
        "body_clean": body_clean,
        "text": text,
        "language_guess": guess_language(text)
    })

df_work = pd.DataFrame(cleaned_rows)

print("Distribusi bahasa hasil deteksi:")
display(
    df_work["language_guess"]
    .value_counts()
    .rename_axis("language")
    .reset_index(name="articles")
)

print("\nArtikel non-Inggris atau belum terdeteksi:")
display(
    df_work.loc[
        df_work["language_guess"] != "en",
        ["original_id", "title", "language_guess", "split"]
    ]
)

Distribusi bahasa hasil deteksi:


,language,articles
0,en,100
1,id,16
2,zh-cn,4



Artikel non-Inggris atau belum terdeteksi:


,original_id,title,language_guess,split
1,6b505436351b490f0b76aa84ceec89ee515ea2e9194d54...,Diplomasi Sukan Anwar Letakkan Malaysia Kembal...,id,train
8,866f27f552b2c710d71168e2b3376065f3bc082d2254fd...,PTT Synergy beli 4 kilang dalam pembinaan bern...,id,train
11,03e3b4d767950fac90f23f15fae288b322157c95a0dcba...,马股周二情绪稍微乐观，跟随区域市场积极走势，综指小幅收涨,zh-cn,train
13,973833e3555ce9b446840412da3ea7be6a4491bc639d37...,"FBM KLCI Susut 15 Mata, Saham Bank & Utiliti T...",id,train
25,6977006b36114fc2cb2bbddf3f738004a424ad76a08a04...,PTT Synergy beli 4 kilang dalam pembinaan bern...,id,train
29,6f19f0f3694cd38f0aae25f796aa48625524a3724bf9ef...,Tengku Ampuan Pahang rasmi Kejohanan Golf Wani...,id,train
30,c1b966b5d0861f7cc203a07ceccf6894aa8a1c653e8d2b...,Warga Palestin diculik: Waran tangkap terhadap...,id,train
44,334ea966eef49c673728228cd22c24c3f6f2d44fe170c2...,马股随隔夜美股反弹，但投资者仍保持谨慎，综指仅微幅收高,zh-cn,train
62,d7e5efa6156d2dd2991d692e5578a2e49e0669894d16dd...,DNB lengkapkan pembiayaan RM5.2 bilion bagi me...,id,train
63,e113277d60ad7eb6ef383bbfc21a786aaf60b35d20e88f...,AHB kekalkan agihan pendapatan lima sen seunit...,id,train


Konversi anotasi ke format spaCy

In [17]:
records = []
unresolved = []
audit = []

english_df = df_work[
    df_work["language_guess"] == "en"
].copy()

for _, row in english_df.iterrows():
    raw_json = str(row["entities_json"]).strip()

    # Baris kosong tidak dianggap otomatis sebagai artikel negatif
    if not raw_json:
        audit.append({
            "original_id": row["original_id"],
            "action": "NEEDS_MANUAL_ANNOTATION",
            "reason": "entities_json kosong"
        })
        continue

    title = row["title"]
    body = row["body"]

    title_clean, title_map, title_tags = (
        strip_html_with_mapping(title)
    )

    body_clean, body_map, body_tags = (
        strip_html_with_mapping(body)
    )

    full_text = title_clean + "\n" + body_clean
    doc = nlp.make_doc(full_text)

    try:
        annotations = json.loads(raw_json)

        if not isinstance(annotations, list):
            raise ValueError("entities_json harus berupa list")

    except Exception as exc:
        unresolved.append({
            "original_id": row["original_id"],
            "reason": f"JSON tidak valid: {exc}"
        })
        continue

    spans = []
    row_problems = []

    for ent in annotations:
        field = ent.get("field")

        if field == "title":
            source = title
            clean = title_clean
            mapping = title_map
            hidden = title_tags
            offset = 0

        elif field == "body":
            source = body
            clean = body_clean
            mapping = body_map
            hidden = body_tags
            offset = len(title_clean) + 1

        else:
            row_problems.append(
                f"Field tidak dikenal: {field}"
            )
            continue

        start = int(ent["start_char"])
        end = int(ent["end_char"])
        entity_text = ent["text"]
        label = ent["label"]

        if not (0 <= start < end <= len(source)):
            row_problems.append(
                f"Offset di luar teks: {entity_text}"
            )
            continue

        if source[start:end] != entity_text:
            row_problems.append(
                f"Teks anotasi tidak cocok: {entity_text}"
            )
            continue

        # Entitas di atribut HTML tidak menjadi teks artikel yang terlihat
        if any(hidden[start:end]):
            audit.append({
                "original_id": row["original_id"],
                "action": "REMOVE_HTML_ATTRIBUTE_SPAN",
                "reason": entity_text
            })
            continue

        # US pada US$... bukan entitas GPE
        if (
            entity_text == "US"
            and label == "GPE"
            and source[end:end + 1] == "$"
        ):
            audit.append({
                "original_id": row["original_id"],
                "action": "REMOVE_FALSE_GPE",
                "reason": "US$ adalah notasi mata uang"
            })
            continue

        new_start = mapping[start]
        new_end = mapping[end]

        if clean[new_start:new_end] != entity_text:
            row_problems.append(
                f"Offset hasil pembersihan tidak cocok: {entity_text}"
            )
            continue

        start_full = new_start + offset
        end_full = new_end + offset

        span = doc.char_span(
            start_full,
            end_full,
            label=label,
            alignment_mode="strict"
        )

        if span is None:
            row_problems.append(
                f"Tidak sejajar dengan token: {entity_text}"
            )
            continue

        spans.append((start_full, end_full, label))

    spans.sort()

    # spaCy NER tidak mendukung rentang entitas yang bertumpang tindih
    for previous, current in zip(spans, spans[1:]):
        if current[0] < previous[1]:
            row_problems.append(
                "Rentang entitas bertumpang tindih"
            )

    if row_problems:
        unresolved.append({
            "original_id": row["original_id"],
            "reason": "; ".join(row_problems)
        })
        continue

    # Jika semua saran otomatis terbuang, perlu review manusia
    if annotations and not spans:
        audit.append({
            "original_id": row["original_id"],
            "action": "NEEDS_MANUAL_REVIEW",
            "reason": "Tidak ada span yang tersisa"
        })
        continue

    records.append({
        "original_id": row["original_id"],
        "split": row["split"].strip().lower(),
        "text": full_text,
        "spans": spans
    })

print("Artikel siap diproses:", len(records))
print("Artikel yang masih bermasalah:", len(unresolved))
print("Catatan audit:", len(audit))

if unresolved:
    display(pd.DataFrame(unresolved).head(20))

Artikel siap diproses: 100
Artikel yang masih bermasalah: 0
Catatan audit: 46


TRAINING DATA

In [18]:
train_records = [
    r for r in records if r["split"] == "train"
]

val_records = [
    r for r in records if r["split"] == "validation"
]

test_records = [
    r for r in records if r["split"] == "test"
]

split_ids = {
    name: {r["original_id"] for r in data}
    for name, data in [
        ("train", train_records),
        ("validation", val_records),
        ("test", test_records)
    ]
}

assert split_ids["train"].isdisjoint(split_ids["validation"])
assert split_ids["train"].isdisjoint(split_ids["test"])
assert split_ids["validation"].isdisjoint(split_ids["test"])

labels = sorted({
    label
    for row in records
    for _, _, label in row["spans"]
})

print("Train:", len(train_records))
print("Validation:", len(val_records))
print("Test:", len(test_records))
print("Label:", labels)

Train: 69
Validation: 17
Test: 14
Label: ['DATE', 'EVENT', 'GPE', 'ORG', 'PERSON']


In [19]:
import spacy
import random
import json
import os
import shutil
import pandas as pd

from spacy.symbols import ORTH
from spacy.util import compile_infix_regex
from spacy.training import Example
from spacy.util import minibatch

SEED = 42
BASE_MODEL = "en_core_web_sm"
MODEL_DIR = "/content/ner_custom_v1"

def configure_tokenizer(nlp):
    infixes = list(nlp.Defaults.infixes)

    # Memisahkan pola seperti end-2027
    pattern = r"(?<=[A-Za-z])-(?=\d)"

    if pattern not in infixes:
        infixes.append(pattern)

    nlp.tokenizer.infix_finditer = (
        compile_infix_regex(infixes).finditer
    )

    # Memisahkan Sept dan titik agar rentang tanggal konsisten
    if "Sept." not in nlp.tokenizer.rules:
        nlp.tokenizer.add_special_case(
            "Sept.",
            [
                {ORTH: "Sept"},
                {ORTH: "."}
            ]
        )

    return nlp


def load_configured_nlp(model_name):
    nlp = spacy.load(model_name)
    configure_tokenizer(nlp)
    return nlp


# tokenizer ini untuk konversi anotasi dan training
nlp = load_configured_nlp(BASE_MODEL)
tokenizer = nlp

print("Model:", BASE_MODEL)
print("Komponen:", nlp.pipe_names)

print([
    token.text
    for token in nlp.make_doc(
        "Rates may change by end-2027, including 29 Sept."
    )
])

Model: en_core_web_sm
Komponen: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
['Rates', 'may', 'change', 'by', 'end', '-', '2027', ',', 'including', '29', 'Sept.']


Sel 2 — Fungsi evaluasi exact-span

In [20]:
def calculate_scores(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn
    }


def evaluate_ner(nlp, data):
    total_tp = total_fp = total_fn = 0
    per_label = {}
    errors = []

    for row in data:
        doc = nlp(row["text"])

        gold = set(row["spans"])
        pred = {
            (ent.start_char, ent.end_char, ent.label_)
            for ent in doc.ents
        }

        correct = gold & pred
        false_positive = pred - gold
        false_negative = gold - pred

        total_tp += len(correct)
        total_fp += len(false_positive)
        total_fn += len(false_negative)

        for _, _, label in gold | pred:
            if label not in per_label:
                per_label[label] = [0, 0, 0]

            per_label[label][0] += sum(
                x[2] == label for x in correct
            )
            per_label[label][1] += sum(
                x[2] == label for x in false_positive
            )
            per_label[label][2] += sum(
                x[2] == label for x in false_negative
            )

        if false_positive or false_negative:
            errors.append({
                "original_id": row["original_id"],
                "false_positive": [
                    {
                        "text": row["text"][s:e],
                        "label": label,
                        "start": s,
                        "end": e
                    }
                    for s, e, label in sorted(false_positive)
                ],
                "false_negative": [
                    {
                        "text": row["text"][s:e],
                        "label": label,
                        "start": s,
                        "end": e
                    }
                    for s, e, label in sorted(false_negative)
                ]
            })

    overall = calculate_scores(
        total_tp, total_fp, total_fn
    )

    label_metrics = {
        label: calculate_scores(*counts)
        for label, counts in per_label.items()
    }

    return {
        "overall": overall,
        "by_label": label_metrics
    }, errors

Sel 3 — Training atau adaptasi model

In [21]:
# Memastikan sel konversi dan pembagian data sudah dijalankan ulang.
print("Train:", len(train_records))
print("Validation:", len(val_records))
print("Test:", len(test_records))

assert train_records, "Data train kosong."
assert val_records, "Data validation kosong."

# Memastikan tidak ada artikel yang bocor antar-split
split_ids = {
    name: {str(r["original_id"]) for r in data}
    for name, data in [
        ("train", train_records),
        ("validation", val_records),
        ("test", test_records)
    ]
}

assert split_ids["train"].isdisjoint(split_ids["validation"])
assert split_ids["train"].isdisjoint(split_ids["test"])
assert split_ids["validation"].isdisjoint(split_ids["test"])

# Baseline memakai tokenizer yang sama
baseline = load_configured_nlp(BASE_MODEL)

baseline_val, _ = evaluate_ner(
    baseline, val_records
)

print("Baseline — validation")
display(pd.DataFrame([baseline_val["overall"]]))

# Muat model untuk adaptasi
spacy.util.fix_random_seed(SEED)

model = load_configured_nlp(BASE_MODEL)
ner = model.get_pipe("ner")

labels = sorted({
    label
    for row in train_records
    for _, _, label in row["spans"]
})

for label in labels:
    ner.add_label(label)

optimizer = model.resume_training()

EPOCHS = 20
best_f1 = -1.0
history = []

if os.path.exists(MODEL_DIR):
    shutil.rmtree(MODEL_DIR)

for epoch in range(EPOCHS):
    shuffled_train = train_records.copy()
    random.Random(SEED + epoch).shuffle(shuffled_train)

    losses = {}

    with model.select_pipes(enable=["ner"]):
        for batch in minibatch(shuffled_train, size=8):
            examples = []

            for row in batch:
                doc = model.make_doc(row["text"])

                example = Example.from_dict(
                    doc,
                    {"entities": row["spans"]}
                )
                examples.append(example)

            model.update(
                examples,
                sgd=optimizer,
                drop=0.2,
                losses=losses
            )

    # Memilih model berdasarkan validation, bukan test
    val_metrics, _ = evaluate_ner(
        model, val_records
    )

    val_f1 = val_metrics["overall"]["f1"]

    history.append({
        "epoch": epoch + 1,
        "ner_loss": losses.get("ner", 0.0),
        "validation_precision":
            val_metrics["overall"]["precision"],
        "validation_recall":
            val_metrics["overall"]["recall"],
        "validation_f1": val_f1
    })

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Loss: {losses.get('ner', 0.0):.3f} | "
        f"Validation F1: {val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        model.to_disk(MODEL_DIR)

print("\nBest validation F1:", best_f1)
print("Model tersimpan:", MODEL_DIR)

history_df = pd.DataFrame(history)
display(history_df.tail())

Train: 69
Validation: 17
Test: 14
Baseline — validation


,precision,recall,f1,tp,fp,fn
0,0.178455,0.602941,0.275399,164,755,108


/usr/local/lib/python3.13/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Haze keeps air quality unhealthy across most of S'..." with entities "[(76, 85, 'GPE'), (172, 181, 'GPE'), (268, 277, 'G...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(


Epoch 01 | Loss: 3600.033 | Validation F1: 0.5369
Epoch 02 | Loss: 1482.476 | Validation F1: 0.7853
Epoch 03 | Loss: 964.091 | Validation F1: 0.8821
Epoch 04 | Loss: 656.035 | Validation F1: 0.9208
Epoch 05 | Loss: 411.732 | Validation F1: 0.9497
Epoch 06 | Loss: 296.168 | Validation F1: 0.9374
Epoch 07 | Loss: 199.522 | Validation F1: 0.9690
Epoch 08 | Loss: 126.354 | Validation F1: 0.9708
Epoch 09 | Loss: 83.841 | Validation F1: 0.9673
Epoch 10 | Loss: 71.107 | Validation F1: 0.9709
Epoch 11 | Loss: 44.483 | Validation F1: 0.9746
Epoch 12 | Loss: 46.054 | Validation F1: 0.9709
Epoch 13 | Loss: 38.079 | Validation F1: 0.9745
Epoch 14 | Loss: 24.327 | Validation F1: 0.9746
Epoch 15 | Loss: 25.800 | Validation F1: 0.9673
Epoch 16 | Loss: 18.166 | Validation F1: 0.9654
Epoch 17 | Loss: 19.973 | Validation F1: 0.9709
Epoch 18 | Loss: 20.037 | Validation F1: 0.9782
Epoch 19 | Loss: 21.739 | Validation F1: 0.9675
Epoch 20 | Loss: 12.229 | Validation F1: 0.9676

Best validation F1: 0.9781818

,epoch,ner_loss,validation_precision,validation_recall,validation_f1
15,16,18.166170,0.956679,0.974265,0.965392
16,17,19.972914,0.960432,0.981618,0.970909
17,18,20.036627,0.967626,0.988971,0.978182
18,19,21.738764,0.950355,0.985294,0.967509
19,20,12.228906,0.947183,0.988971,0.967626


Sel 4 — Menyiapkan test set untuk review

In [22]:
from google.colab import files

review_rows = []

for row in test_records:
    spans_json = [
        {
            "text": row["text"][start:end],
            "label": label,
            "start": start,
            "end": end
        }
        for start, end, label in row["spans"]
    ]

    review_rows.append({
        "original_id": row["original_id"],
        "text": row["text"],
        "gold_spans_json": json.dumps(
            spans_json, ensure_ascii=False
        ),
        "review_status": "NOT_REVIEWED",
        "reviewer_notes": ""
    })

test_review_df = pd.DataFrame(review_rows)

review_path = "/content/test_gold_review.csv"

test_review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Jumlah artikel untuk review:", len(test_review_df))
files.download(review_path)

Jumlah artikel untuk review: 14


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
from google.colab import files
import pandas as pd
import json
import os
import shutil

# ==================================================
# 1. UPLOAD CSV TEST YANG SUDAH KAMU REVIEW
# ==================================================

uploaded_review = files.upload()

review_file = next(
    (
        name for name in uploaded_review
        if name.lower().endswith(".csv")
    ),
    None
)

assert review_file is not None, (
    "Silakan upload test_gold_review_reviewed.csv"
)

reviewed_df = pd.read_csv(
    review_file,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

required_cols = {
    "original_id",
    "text",
    "gold_spans_json",
    "review_status"
}

assert required_cols.issubset(reviewed_df.columns), (
    f"Kolom kurang: {required_cols - set(reviewed_df.columns)}"
)

assert (
    reviewed_df["review_status"]
    .str.strip()
    .str.upper()
    .eq("REVIEWED")
    .all()
), "Masih ada artikel yang belum direview."

assert reviewed_df["original_id"].is_unique, (
    "Ada original_id duplikat dalam CSV test."
)

assert len(reviewed_df) >= 10, (
    "Test set harus memuat minimal 10 artikel."
)

print("File:", review_file)
print("Jumlah artikel:", len(reviewed_df))
print("Status review: REVIEWED")


# ==================================================
# 2. VALIDASI LABEL DAN RENTANG KARAKTER
# ==================================================

reviewed_test_records = []

# Gunakan tokenizer yang sama dengan preprocessing/training
eval_tokenizer = load_configured_nlp(BASE_MODEL)

for _, row in reviewed_df.iterrows():
    text = row["text"]
    annotations = json.loads(row["gold_spans_json"])

    assert isinstance(annotations, list), (
        f"Format JSON salah: {row['original_id']}"
    )

    spans = []
    doc = eval_tokenizer.make_doc(text)

    for ent in annotations:
        start = int(ent["start"])
        end = int(ent["end"])
        entity_text = str(ent["text"])
        label = str(ent["label"])

        assert 0 <= start < end <= len(text), (
            f"Offset tidak valid: {row['original_id']} "
            f"/ {entity_text}"
        )

        assert text[start:end] == entity_text, (
            f"Teks tidak cocok dengan offset: "
            f"{row['original_id']} / {entity_text}"
        )

        span = doc.char_span(
            start,
            end,
            label=label,
            alignment_mode="strict"
        )

        assert span is not None, (
            f"Rentang tidak sejajar dengan tokenizer: "
            f"{row['original_id']} / {entity_text}. "
            f"Periksa konfigurasi tokenizer."
        )

        spans.append((start, end, label))

    spans.sort()

    for previous, current in zip(spans, spans[1:]):
        assert current[0] >= previous[1], (
            f"Entitas overlap pada {row['original_id']}"
        )

    reviewed_test_records.append({
        "original_id": str(row["original_id"]),
        "text": text,
        "spans": spans
    })

print("\nValidasi selesai.")
print("Artikel test:", len(reviewed_test_records))
print(
    "Jumlah entitas:",
    sum(len(r["spans"]) for r in reviewed_test_records)
)


# ==================================================
# 3. LOAD BASELINE DAN MODEL HASIL TRAINING
# ==================================================

assert os.path.exists(MODEL_DIR), (
    f"Model tidak ditemukan: {MODEL_DIR}. "
    "Jalankan training terlebih dahulu."
)

baseline = load_configured_nlp(BASE_MODEL)
final_model = load_configured_nlp(MODEL_DIR)

print("Baseline:", BASE_MODEL)
print("Model akhir:", MODEL_DIR)


# ==================================================
# 4. EVALUASI PADA TEST SET
# ==================================================

baseline_test, baseline_errors = evaluate_ner(
    baseline,
    reviewed_test_records
)

final_test, final_errors = evaluate_ner(
    final_model,
    reviewed_test_records
)

comparison = pd.DataFrame([
    {
        "Model": "Baseline pretrained",
        **baseline_test["overall"]
    },
    {
        "Model": "Model hasil training",
        **final_test["overall"]
    }
])

print("\nHASIL EVALUASI TEST — EXACT SPAN + LABEL")
display(comparison)


# ==================================================
# 5. METRIK PER LABEL
# ==================================================

print("METRIK PER LABEL — BASELINE")
display(
    pd.DataFrame(baseline_test["by_label"]).T
)

print("METRIK PER LABEL — MODEL HASIL TRAINING")
display(
    pd.DataFrame(final_test["by_label"]).T
)


# ==================================================
# 6. ANALISIS KESALAHAN
# ==================================================

error_df = pd.DataFrame(final_errors)

print("Jumlah artikel dengan prediksi yang berbeda dari label acuan:",
      len(error_df))

display(error_df.head(10))


# ==================================================
# 7. SIMPAN HASIL EVALUASI
# ==================================================

OUTPUT_DIR = "/content/ner_model_deliverables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(
    f"{OUTPUT_DIR}/evaluation_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "baseline_test": baseline_test,
            "final_test": final_test,
            "matching": "exact span and entity label",
            "test_review_status": "REVIEWED"
        },
        f,
        ensure_ascii=False,
        indent=2
    )

error_df.to_csv(
    f"{OUTPUT_DIR}/error_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

if "history_df" in globals():
    history_df.to_csv(
        f"{OUTPUT_DIR}/training_history.csv",
        index=False,
        encoding="utf-8-sig"
    )


# ==================================================
# 8. ZIP MODEL DAN HASIL EVALUASI
# ==================================================

model_zip = shutil.make_archive(
    "/content/ner_custom_v1",
    "zip",
    root_dir=MODEL_DIR
)

results_zip = shutil.make_archive(
    "/content/ner_model_deliverables",
    "zip",
    root_dir=OUTPUT_DIR
)

print("\nSELESAI!")
print("Model:", model_zip)
print("Hasil evaluasi:", results_zip)

files.download(model_zip)
files.download(results_zip)

Saving test_gold_review_reviewed.csv to test_gold_review_reviewed.csv
File: test_gold_review_reviewed.csv
Jumlah artikel: 14
Status review: REVIEWED

Validasi selesai.
Artikel test: 14
Jumlah entitas: 410
Baseline: en_core_web_sm
Model akhir: /content/ner_custom_v1

HASIL EVALUASI TEST — EXACT SPAN + LABEL


,Model,precision,recall,f1,tp,fp,fn
0,Baseline pretrained,0.179545,0.578049,0.273988,237,1083,173
1,Model hasil training,0.950588,0.985366,0.967665,404,21,6


METRIK PER LABEL — BASELINE


,precision,recall,f1,tp,fp,fn
MONEY,0.000000,0.000000,0.000000,0.0,277.0,0.0
CARDINAL,0.000000,0.000000,0.000000,0.0,9513.0,0.0
DATE,0.332992,0.673652,0.445680,9419.0,18867.0,4563.0
PERCENT,0.000000,0.000000,0.000000,0.0,7766.0,0.0
PERSON,0.029265,0.598425,0.055800,76.0,2521.0,51.0
ORG,0.097337,0.430839,0.158797,2361.0,21895.0,3119.0
GPE,0.556996,0.801110,0.657114,6787.0,5398.0,1685.0
PRODUCT,0.000000,0.000000,0.000000,0.0,15.0,0.0
NORP,0.000000,0.000000,0.000000,0.0,258.0,0.0
LOC,0.000000,0.000000,0.000000,0.0,34.0,0.0


METRIK PER LABEL — MODEL HASIL TRAINING


...


SELESAI!
Model: /content/ner_custom_v1.zip
Hasil evaluasi: /content/ner_model_deliverables.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>